# VL 8 — Abschlussprojekt: Schiffe versenken (komplett)

## Lernziele

- Alle bisherigen Konzepte in einem Gesamtprojekt anwenden
- Code modular strukturieren (mehrere Dateien)
- Systematisch testen und Edge Cases behandeln

**Ziel dieser Vorlesung:** Alle Teilstücke aus VL 1–7 werden zu einem vollständig spielbaren Konsolenspiel zusammengebaut.

---
## Block 1: Architektur & Zusammenführung (45 min)

### Gesamtstruktur planen

Unser Spiel besteht aus drei Modulen:

```
Projektstruktur:
├── main.py            ← Spielschleife + Menü
├── spiellogik.py      ← Spielfeld, Schüsse, Auswertung, Platzierung
└── datei.py           ← Speichern/Laden
```

| Modul | Funktionen | VL |
|-------|------------|----|
| `spiellogik.py` | `erstelle_spielfeld()` | VL 5 |
| | `spielfeld_ausgeben(spielfeld)` | VL 5 |
| | `eingabe_lesen()` | VL 5 + 7 |
| | `schuss_auswerten(spielfeld, schiffe, koordinate)` | VL 5 |
| | `platziere_schiffe()` | VL 6 |
| | `spiel_beendet(schiffe)` | **NEU** |
| `datei.py` | `spielstand_speichern(dateiname, spielfeld, statistik)` | VL 6 |
| | `spielstand_laden(dateiname)` | VL 6 |
| | `lade_sicher(dateiname)` | VL 7 |
| `main.py` | `zeige_menue()` | VL 7 |
| | Menüschleife + Spielschleife | VL 2 + 7 |

### Funktions-Übersicht

Hier nochmal alle Funktionen mit ihren Signaturen:

```python
# spiellogik.py
def erstelle_spielfeld()                              # → 2D-Liste (5×5 mit "~")
def spielfeld_ausgeben(spielfeld)                     # → None (gibt auf Konsole aus)
def eingabe_lesen()                                   # → (zeile, spalte) oder None
def schuss_auswerten(spielfeld, schiffe, koordinate)  # → "treffer"/"wasser"/"versenkt"
def platziere_schiffe()                               # → Dictionary {Name: [(z,s), ...]}
def spiel_beendet(schiffe)                            # → True/False

# datei.py
def spielstand_speichern(dateiname, spielfeld, statistik)  # → None
def spielstand_laden(dateiname)                            # → spielfeld, statistik
def lade_sicher(dateiname)                                 # → (spielfeld, statistik) oder None

# main.py
def zeige_menue()                                     # → None (gibt Menü aus)
```

**Was fehlt?** Die Funktion `spiel_beendet(schiffe)` — diese wird heute implementiert.

### Zusammenspiel der Funktionen

Der Spielablauf sieht so aus:

```
Menüschleife:
│
├── "1" Neues Spiel:
│   ├── erstelle_spielfeld()
│   ├── platziere_schiffe()
│   └── Spielschleife:
│       ├── spielfeld_ausgeben()
│       ├── eingabe_lesen()        ← try/except
│       ├── schuss_auswerten()
│       ├── Statistik aktualisieren
│       └── spiel_beendet()?       ← Abbruchbedingung
│
├── "2" Laden:
│   └── lade_sicher()              ← try/except
│
├── "3" Regeln
│
└── "4" Beenden → break
```

---
## Block 2 + 3: Praxis — Das komplette Spiel (90 min)

In den folgenden Aufgaben baust du alle Teilstücke zum fertigen Spiel zusammen. Nutze die Funktionen aus den vorherigen Vorlesungen — kopiere sie hierher oder definiere sie neu.

### Aufgabe 1: Bestandsaufnahme

Sammle alle Funktionen aus VL 1–7, die du für das Spiel brauchst. Definiere sie in den folgenden Zellen. Welche existieren bereits, welche fehlen?

Schreibe die Signatur (nur `def`-Zeile + `pass`) für die fehlende Funktion `spiel_beendet(schiffe)` auf.

**Hinweis:** Du kannst die Funktionen aus den vorherigen Notebooks hierher kopieren.

<details>
<summary>💡 Tipp</summary>

Definiere alle Funktionen in einer oder mehreren Code-Zellen, bevor du sie weiter unten verwendest. Die Reihenfolge:
1. Hilfsfunktionen (erstelle_spielfeld, spielfeld_ausgeben, eingabe_lesen)
2. Spiellogik (schuss_auswerten, platziere_schiffe, spiel_beendet)
3. Dateiverwaltung (spielstand_speichern, spielstand_laden, lade_sicher)
4. Menü (zeige_menue)
</details>

### Aufgabe 2: `spiel_beendet(schiffe)` implementieren

Schreibe die Funktion `spiel_beendet(schiffe)`. Sie bekommt das Schiffe-Dictionary und prüft, ob alle Schiffe versenkt sind.

**Logik:** Iteriere über alle Schiffe im Dictionary. Wenn **alle** Positionslisten leer sind → `True` (Spiel beendet). Wenn mindestens eine Liste noch Positionen hat → `False`.

Teste mit:
- `{"Kreuzer": [], "U-Boot": []}` → `True`
- `{"Kreuzer": [(0, 1)], "U-Boot": []}` → `False`

<details>
<summary>💡 Tipp</summary>

```python
def spiel_beendet(schiffe):
    for name, positionen in schiffe.items():
        if len(positionen) > 0:
            return False
    return True
```
</details>

### Aufgabe 3: Spielschleife zusammenbauen

Schreibe die Hauptspielschleife:

1. `erstelle_spielfeld()` aufrufen
2. `platziere_schiffe()` aufrufen (zufällig)
3. Statistik-Dictionary initialisieren: `{"treffer": 0, "wasser": 0, "versenkt": 0}`
4. Geschossene Koordinaten als Set: `geschossen = set()`
5. `while not spiel_beendet(schiffe):`
   - `spielfeld_ausgeben(spielfeld)`
   - `koordinate = eingabe_lesen()` (mit try/except)
   - Falls `None`: `continue`
   - Falls schon geschossen: Meldung + `continue`
   - `schuss_auswerten()` aufrufen und Statistik aktualisieren
   - Koordinate zum Set hinzufügen
6. Nach der Schleife: Spielfeld + Statistik ausgeben

<details>
<summary>💡 Tipp</summary>

```python
spielfeld = erstelle_spielfeld()
schiffe = platziere_schiffe()
statistik = {"treffer": 0, "wasser": 0, "versenkt": 0}
geschossen = set()

while not spiel_beendet(schiffe):
    spielfeld_ausgeben(spielfeld)
    koordinate = eingabe_lesen()
    
    if koordinate is None:
        continue
    
    if koordinate in geschossen:
        print("Bereits geschossen!")
        continue
    
    geschossen.add(koordinate)
    ergebnis = schuss_auswerten(spielfeld, schiffe, koordinate)
    statistik[ergebnis] = statistik[ergebnis] + 1
    print(f"Ergebnis: {ergebnis}")

print("Alle Schiffe versenkt!")
spielfeld_ausgeben(spielfeld)
for key, value in statistik.items():
    print(f"{key}: {value}")
```
</details>

### Aufgabe 4: Menü integrieren

Baue die Menüschleife aus VL 7 ein und verbinde sie mit der Spielschleife:

- `"1"` → Spielschleife aus Aufgabe 3 starten
- `"2"` → Spielstand laden mit `lade_sicher()` (try/except). Falls erfolgreich: Spielfeld ausgeben
- `"3"` → Spielregeln ausgeben (kurzer Text)
- `"4"` → `break` — Programm beenden
- `_` → `"Ungültige Eingabe!"`

Nutze `match`/`case` für die Menüsteuerung.

<details>
<summary>💡 Tipp</summary>

```python
while True:
    zeige_menue()
    wahl = input("Deine Wahl: ")
    
    match wahl:
        case "1":
            # Spielschleife aus Aufgabe 3 hierher
            pass
        case "2":
            ergebnis = lade_sicher("spielstand.csv")
            if ergebnis:
                spielfeld, statistik = ergebnis
                spielfeld_ausgeben(spielfeld)
        case "3":
            print("Versenke alle Schiffe auf dem 5×5 Spielfeld!")
        case "4":
            print("Auf Wiedersehen!")
            break
        case _:
            print("Ungültige Eingabe!")
```
</details>

### Aufgabe 5: Spielstand-Integration

Erweitere das Spiel um Spielstand-Funktionalität:

- **Nach Spielende:** Frage den Spieler ob der Spielstand gespeichert werden soll
- **Beim Laden (Option 2):** Lade den Spielstand und setze das Spiel an der gespeicherten Stelle fort

<details>
<summary>💡 Tipp</summary>

Nach der Spielschleife:
```python
antwort = input("Spielstand speichern? (j/n): ")
if antwort == "j":
    spielstand_speichern("spielstand.csv", spielfeld, statistik)
    print("Gespeichert!")
```
</details>

### Aufgabe 6: Edge Cases testen

Teste dein Spiel systematisch mit folgenden Szenarien:

1. **Doppelter Schuss:** Gleiche Koordinate zweimal eingeben → `"Bereits geschossen!"`
2. **Ungültige Eingabe:** z.B. `"Z9"`, `""`, `"123"` → Fehlermeldung, kein Absturz
3. **Nicht vorhandene Datei laden:** Option 2 wählen ohne Spielstand → Fehlermeldung
4. **Alle Schiffe versenken:** Spiel bis zum Ende spielen → Glückwunsch + Statistik

Dokumentiere in der folgenden Zelle, was du getestet hast und ob es funktioniert.

### Aufgabe 7: Reflexion

Ordne jede Funktion im Spiel einer Vorlesung zu. Schreibe für jede Funktion auf, welche Python-Konzepte darin stecken.

Beispiel:
```
erstelle_spielfeld() → VL 5 (Funktionen) + VL 3 (Listen)
  Konzepte: def/return, verschachtelte Listen, for-Schleife, append()
```

Schreibe deine Zuordnung in die folgende Markdown-Zelle oder Code-Zelle (als Kommentare).